# 层次聚类：一棵树看清数据的层级结构
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/聚类 | 源文件：层次聚类.py | 核心功能：凝聚聚类、linkage 方法对比、树状图、自动确定簇数

## 概述

层次聚类构建一棵聚类树（树状图/Dendrogram），从图上可以清楚看到数据在不同粒度上的分组结构。凝聚层次聚类（自底向上）：每个点初始为一个簇，逐步合并最相似的两个簇，直到所有点合并为一个簇。

与 KMeans 的关键区别：KMeans 只给出一个分组结果，层次聚类给出**所有可能的分组**——通过在不同高度"切割"树状图，可以得到不同数量的簇。

脚本对比了四种 linkage 方法，演示了树状图的解读和 distance_threshold 自动确定簇数。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
# --- 导入库 ---
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

## 数学原理

### 1. 层次聚类的两种策略

**凝聚式**（Agglomerative，自底向上）：每个样本初始为一个簇，逐步合并最近的两个簇。

**分裂式**（Divisive，自顶向下）：所有样本初始为一个簇，逐步分裂。

sklearn 使用凝聚式。

### 2. 簇间距离度量（Linkage）

**代码对应**：`AgglomerativeClustering(linkage="ward")` 中的 linkage 参数。

设簇 $A$ 和 $B$ 合并为 $A \cup B$，常用的簇间距离：

**单链接**（Single Linkage）：$d(A, B) = \min_{\mathbf{a} \in A, \mathbf{b} \in B}\|\mathbf{a} - \mathbf{b}\|$

**全链接**（Complete Linkage）：$d(A, B) = \max_{\mathbf{a} \in A, \mathbf{b} \in B}\|\mathbf{a} - \mathbf{b}\|$

**平均链接**（Average Linkage）：$d(A, B) = \frac{1}{|A||B|}\sum_{\mathbf{a} \in A}\sum_{\mathbf{b} \in B}\|\mathbf{a} - \mathbf{b}\|$

**Ward 链接**：合并后 WCSS 增量最小的两个簇：

$$\Delta(A, B) = \frac{|A||B|}{|A|+|B|}\|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|^2$$

Ward 方法倾向于产生大小相近的球形簇，效果通常最好。

### 3. 合并策略的 Lance-Williams 递推

所有链接方法可以统一为 Lance-Williams 递推公式：

$$d(A \cup B, C) = \alpha_A d(A, C) + \alpha_B d(B, C) + \beta d(A, B) + \gamma|d(A, C) - d(B, C)|$$

不同链接对应不同的 $(\alpha_A, \alpha_B, \beta, \gamma)$ 参数。

### 4. 树状图（Dendrogram）

树状图记录了每次合并的距离。通过设置距离阈值 $h$，可以在任意层级"切割"树状图得到指定数量的簇。

数学上，层次聚类产生一个**偏序集**（partially ordered set），树状图是其可视化。不同切割位置对应不同的簇数。

### 5. 时间与空间复杂度

- **时间复杂度**：$O(n^3)$（朴素实现），$O(n^2\log n)$（使用优先队列）
- **空间复杂度**：$O(n^2)$ 存储距离矩阵

因此层次聚类不适合大数据集（$n > 10^4$）。

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y_true = make_blobs(n_samples=150, centers=4, cluster_std=1.0, random_state=42)
X = StandardScaler().fit_transform(X)

### 2. 凝聚层次聚类（AgglomerativeClustering）

运行 2. 凝聚层次聚类（AgglomerativeClustering） 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 凝聚层次聚类 ===")
# linkage: "ward"(最小化方差), "complete"(最大距离), "average"(平均距离), "single"(最小距离)
for linkage_method in ["ward", "complete", "average", "single"]:
    ac = AgglomerativeClustering(n_clusters=4, linkage=linkage_method)
    labels = ac.fit_predict(X)
    sil = silhouette_score(X, labels)
    ari = adjusted_rand_score(y_true, labels)
# --- 输出结果 ---
    print(f"  linkage={linkage_method:>8}: Silhouette={sil:.4f}, ARI={ari:.4f}")

### 3. Ward 链接法详细分析

运行 3. Ward 链接法详细分析 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Ward 链接法 (n_clusters=4) ===")
ac_ward = AgglomerativeClustering(n_clusters=4, linkage="ward")
labels_ward = ac_ward.fit_predict(X)
print(f"簇标签分布: {np.bincount(labels_ward)}")
print(f"Silhouette: {silhouette_score(X, labels_ward):.4f}")

### 4. 不同簇数对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 不同 n_clusters 对比 ===")
for k in range(2, 8):
    ac_k = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels_k = ac_k.fit_predict(X)
    sil = silhouette_score(X, labels_k)
# --- 输出结果 ---
    print(f"  K={k}: Silhouette={sil:.4f}")

### 5. 树状图（用 scipy）

运行 5. 树状图（用 scipy） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 树状图（Dendrogram）===")
# 计算层次链接矩阵
Z = linkage(X, method="ward")
print(f"链接矩阵形状: {Z.shape}")
print(f"最后 5 次合并:")
for i in range(-5, 0):
    print(f"  合并 {int(Z[i, 0]):>5} 和 {int(Z[i, 1]):>5} → {int(Z[i, 0])+int(Z[i, 1]):>5}, "
# --- 赋值/配置 ---
          f"距离={Z[i, 2]:.4f}")

# 用 fcluster 从链接矩阵切割出簇
for k in [2, 3, 4, 5]:
    labels_f = fcluster(Z, t=k, criterion="maxclust")
    sil = silhouette_score(X, labels_f)
    print(f"  fcluster K={k}: Silhouette={sil:.4f}")

### 6. distance_threshold 自动确定簇数

运行 6. distance_threshold 自动确定簇数 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== distance_threshold 自动切割 ===")
ac_auto = AgglomerativeClustering(n_clusters=None, distance_threshold=3.0, linkage="ward")
labels_auto = ac_auto.fit_predict(X)
n_clusters = len(set(labels_auto))
print(f"阈值=3.0 时自动确定簇数: {n_clusters}")

# 不同阈值
for thresh in [1.0, 2.0, 3.0, 5.0, 10.0]:
    ac_t = AgglomerativeClustering(n_clusters=None, distance_threshold=thresh, linkage="ward")
    labels_t = ac_t.fit_predict(X)
    nc = len(set(labels_t))
    print(f"  阈值={thresh:>5}: 簇数={nc}")

print("\n=== 层次聚类要点 ===")
print("- 凝聚（自底向上）：每个点初始为一个簇，逐步合并最相似的簇")
print("- 分裂（自顶向下）：所有点在一个簇，逐步分裂")
print("- linkage 决定簇间距离的计算方式")
print("- Ward 法：合并后总方差增加最小的两个簇（默认推荐）")
# --- 输出结果 ---
print("- Single 法：容易出现链式效应（chaining）")
print("- 优点：不需要预设 K（可用阈值切割）、可产生树状图")
print("- 缺点：时间复杂度 O(n²~n³)、合并/分裂不可撤销（贪心）")

## 关键代码解释

### 四种 linkage 方法

```python
for linkage_method in ["ward", "complete", "average", "single"]:
    ac = AgglomerativeClustering(n_clusters=4, linkage=linkage_method)
```

- **Ward**：合并后总方差增加最小的两个簇（默认推荐，效果通常最好）
- **Complete**：两簇中距离最远的两个点的距离（紧凑簇）
- **Average**：两簇所有点对的平均距离
- **Single**：两簇中距离最近的两个点的距离（容易链式效应）

### distance_threshold 自动切割

```python
ac_auto = AgglomerativeClustering(n_clusters=None, distance_threshold=3.0, linkage="ward")
```

设置 
_clusters=None + distance_threshold，可以不预设簇数，由阈值自动决定。

## 使用示例

```python
from sklearn.cluster import AgglomerativeClustering
ac = AgglomerativeClustering(n_clusters=4, linkage="ward")
labels = ac.fit_predict(X)
```

## 注意事项

1. **时间复杂度 O(n²~n³)**：不适合大数据集（> 10000 样本）
2. **合并不可撤销**：贪心策略，一旦合并就无法拆分
3. **Ward 只支持欧氏距离**
4. **树状图解读**：纵轴是合并距离，横轴是样本，水平切割得到不同数量的簇

## 总结与延伸

以上代码展示了 **层次聚类** 的完整流程。

**解读要点**：
- 关注 **轮廓系数（Silhouette Score）**，越接近 1 越好
- 观察聚类中心是否与数据分布吻合
- 对比不同 K 值的聚类效果

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **BIRCH**：大规模层次聚类的高效实现
- **CURE**：用代表点替代质心，处理非球形簇
- **谱聚类**：本质上也是一种利用图结构的层次化方法
- **树状图的统计检验**：用 bootstrap 评估聚类的稳定性
